# 2SFS: Two-Stage Few-Shot Adaptation of CLIP

Implements the **2SFS** method (Farina et al., CVPR 2025 / arXiv 2024) reviewed in Section 2.3.3 of the report: a two-stage PEFT approach.

- **Stage 1** — fine-tune *only the LayerNorm parameters* of CLIP's visual encoder (weight + bias of every `ln_1`/`ln_2`/`ln_post`) on the K-shot support set, using a held-out validation split to find the **breakpoint** — the epoch where the model starts to over-specialize on the few-shot classes.
- **Stage 2** — freeze the encoder at the breakpoint checkpoint and train a linear classifier on top of the resulting (adapted) frozen features.

This uses by far the fewest trainable parameters of any method in this comparison — only the LayerNorm affine parameters plus a small linear head.

---

In [14]:
# ── Install required packages ──────────────────────────────────────────────
!pip install -q ftfy regex tqdm
!pip install -q git+https://github.com/openai/CLIP.git
print("\u2705 Installed.")

  Preparing metadata (setup.py) ... done
✅ Installed.


In [15]:
# ── Core imports ───────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import clip
import copy
import numpy as np
import matplotlib.pyplot as plt

import torchvision
import torchvision.transforms as transforms
from torch.utils.data import Subset

import warnings
warnings.filterwarnings('ignore')

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on: {DEVICE.upper()}")

Running on: CUDA


## 1 — Load CLIP and CIFAR-10

In [16]:
print("Loading CLIP model (ViT-B/32)...")
clip_model, clip_preprocess = clip.load("ViT-B/32", device=DEVICE)
clip_model = clip_model.float()   # train in fp32 for stability
clip_model.eval()
print("\u2705 CLIP loaded!")

Loading CLIP model (ViT-B/32)...
✅ CLIP loaded!


In [17]:
cifar_train = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transforms.ToTensor())
raw_cifar10 = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transforms.ToTensor())
CIFAR10_CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

indices_per_class = {c: [] for c in range(10)}
for idx, (_, label) in enumerate(raw_cifar10):
    if len(indices_per_class[label]) < 50:
        indices_per_class[label].append(idx)
    if all(len(v) == 50 for v in indices_per_class.values()):
        break
test_indices = [idx for idxs in indices_per_class.values() for idx in idxs]
test_subset = Subset(raw_cifar10, test_indices)
print(f"Test set: {len(test_subset)} images (50 per class)")

text_prompts = [f"a photo of a {c}" for c in CIFAR10_CLASSES]
text_tokens = clip.tokenize(text_prompts).to(DEVICE)
with torch.no_grad():
    text_features_fixed = clip_model.encode_text(text_tokens)
    text_features_fixed = text_features_fixed / text_features_fixed.norm(dim=-1, keepdim=True)
print(f"Frozen zero-shot text classifier ready: {text_features_fixed.shape}")

Test set: 500 images (50 per class)
Frozen zero-shot text classifier ready: torch.Size([10, 512])


In [18]:
def build_shot_subset(k, seed=0):
    """K images per class from the CIFAR-10 train split."""
    indices = []
    seen = {c: 0 for c in range(10)}
    for idx, (_, label) in enumerate(cifar_train):
        if seen[label] < k:
            indices.append(idx)
            seen[label] += 1
        if all(v == k for v in seen.values()):
            break
    return indices

def preprocess_batch(dataset, indices):
    imgs = torch.stack([clip_preprocess(transforms.ToPILImage()(dataset[i][0])) for i in indices])
    labels = torch.tensor([dataset[i][1] for i in indices])
    return imgs.to(DEVICE), labels.to(DEVICE)

---

## 2 — Stage 1: LayerNorm-only fine-tuning with breakpoint detection

In [19]:
# ── Identify and (un)freeze LayerNorm parameters in the visual encoder ──────
def get_layernorm_param_names(model):
    names = []
    for name, module in model.visual.named_modules():
        if isinstance(module, nn.LayerNorm):
            names.append(name)
    return names

def freeze_all_except_layernorm(model):
    trainable_params = []
    ln_names = set(get_layernorm_param_names(model))
    for name, param in model.visual.named_parameters():
        parent = name.rsplit('.', 1)[0]
        if parent in ln_names:
            param.requires_grad = True
            trainable_params.append(param)
        else:
            param.requires_grad = False
    for param in model.transformer.parameters():   # freeze text encoder entirely
        param.requires_grad = False
    return trainable_params

n_ln_layers = len(get_layernorm_param_names(clip_model))
print(f"Found {n_ln_layers} LayerNorm modules in the visual encoder.")

Found 26 LayerNorm modules in the visual encoder.


In [20]:
# ── Stage 1 training loop with a held-out val split for breakpoint detection ─
def stage1_finetune(k, epochs=15, lr=1e-4):
    all_idx = build_shot_subset(k)

    # Hold out ~20% (min 1 per class where possible) as validation to detect the breakpoint
    if k >= 2:
        val_frac = max(1, k // 5)
        by_class = {c: [] for c in range(10)}
        for i in all_idx:
            by_class[cifar_train[i][1]].append(i)
        val_idx, train_idx = [], []
        for c, idxs in by_class.items():
            val_idx.extend(idxs[:val_frac])
            train_idx.extend(idxs[val_frac:])
    else:
        train_idx, val_idx = all_idx, all_idx   # K=1 edge case: no separate val split possible

    model = copy.deepcopy(clip_model)
    trainable_params = freeze_all_except_layernorm(model)
    optimizer = torch.optim.AdamW(trainable_params, lr=lr)
    criterion = nn.CrossEntropyLoss()

    train_imgs, train_labels = preprocess_batch(cifar_train, train_idx)
    val_imgs, val_labels     = preprocess_batch(cifar_train, val_idx)

    best_val_acc, best_state, breakpoint_epoch = -1, None, 0

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        feats = model.encode_image(train_imgs)
        feats = feats / feats.norm(dim=-1, keepdim=True)
        logits = 100.0 * feats @ text_features_fixed.T
        loss = criterion(logits, train_labels)
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_feats = model.encode_image(val_imgs)
            val_feats = val_feats / val_feats.norm(dim=-1, keepdim=True)
            val_logits = 100.0 * val_feats @ text_features_fixed.T
            val_acc = (val_logits.argmax(dim=1) == val_labels).float().mean().item() * 100

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            breakpoint_epoch = epoch + 1
            best_state = copy.deepcopy({n: p.detach().clone() for n, p in model.visual.named_parameters() if p.requires_grad})

    # Restore the best (breakpoint) LayerNorm weights
    with torch.no_grad():
        for n, p in model.visual.named_parameters():
            if n in best_state:
                p.copy_(best_state[n])

    return model, breakpoint_epoch, best_val_acc, all_idx

---

## 3 — Stage 2: Linear probe on the adapted (LayerNorm-tuned) features

In [21]:
class LinearProbe(nn.Module):
    def __init__(self, input_dim=512, num_classes=10):
        super().__init__()
        self.fc = nn.Linear(input_dim, num_classes)
    def forward(self, x):
        return self.fc(x)

def extract_features_with_model(model, dataset, indices):
    feats = []
    model.eval()
    with torch.no_grad():
        for i in indices:
            pil_img = transforms.ToPILImage()(dataset[i][0])
            inp = clip_preprocess(pil_img).unsqueeze(0).to(DEVICE)
            f = model.encode_image(inp)
            f = f / f.norm(dim=-1, keepdim=True)
            feats.append(f.cpu())
    return torch.cat(feats, dim=0).float()

def stage2_linear_probe(model, train_idx, test_indices_global, epochs=10, lr=1e-3, batch_size=64):
    X_train = extract_features_with_model(model, cifar_train, train_idx)
    y_train = torch.tensor([cifar_train[i][1] for i in train_idx])

    X_test = extract_features_with_model(model, raw_cifar10, test_indices_global)
    y_test = torch.tensor([raw_cifar10[i][1] for i in test_indices_global])

    probe = LinearProbe(X_train.shape[1]).to(DEVICE)
    optimizer = torch.optim.Adam(probe.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(X_train.to(DEVICE), y_train.to(DEVICE)),
        batch_size=batch_size, shuffle=True)

    for epoch in range(epochs):
        probe.train()
        for bx, by in loader:
            optimizer.zero_grad()
            loss = criterion(probe(bx), by)
            loss.backward()
            optimizer.step()

    probe.eval()
    with torch.no_grad():
        preds = probe(X_test.to(DEVICE)).argmax(dim=1).cpu()
    return (preds == y_test).float().mean().item() * 100

---

## 4 — Run the full K-shot sweep

In [22]:
K_VALUES = [1, 5, 10, 50, 100]
results = {'breakpoint_epoch': {}, 'stage1_val_acc': {}, 'final_test_acc': {}, 'n_trainable': {}}

print(f"{'K':>5} {'Breakpoint ep.':>15} {'Stage1 val acc':>15} {'Final test acc':>16}")
print("-" * 58)

for k in K_VALUES:
    adapted_model, bp_epoch, val_acc, all_idx = stage1_finetune(k)
    test_acc = stage2_linear_probe(adapted_model, all_idx, test_indices)

    n_trainable = sum(p.numel() for n, p in adapted_model.visual.named_parameters() if p.requires_grad)

    results['breakpoint_epoch'][k] = bp_epoch
    results['stage1_val_acc'][k]   = val_acc
    results['final_test_acc'][k]   = test_acc
    results['n_trainable'][k]      = n_trainable

    print(f"{k:>5} {bp_epoch:>15} {val_acc:>14.1f}% {test_acc:>15.1f}%")

print("-" * 58)
print(f"\nTrainable LayerNorm parameters (Stage 1): {results['n_trainable'][K_VALUES[0]]:,}")

    K  Breakpoint ep.  Stage1 val acc   Final test acc
----------------------------------------------------------
    1              12          100.0%            42.6%
    5               1           80.0%            47.8%
   10              15           90.0%            74.0%
   50              11           91.0%            90.2%


OutOfMemoryError: CUDA out of memory. Tried to allocate 470.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 319.81 MiB is free. Including non-PyTorch memory, this process has 14.25 GiB memory in use. Of the allocated memory 13.29 GiB is allocated by PyTorch, and 841.30 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [23]:
# ── Plot the few-shot learning curve ─────────────────────────────────────────
plt.figure(figsize=(8, 5.5))
plt.plot(K_VALUES, [results['final_test_acc'][k] for k in K_VALUES], 'o-', linewidth=2, markersize=8, color='#9C27B0')
plt.xlabel('Shots per class (K)')
plt.ylabel('Test Accuracy (%)')
plt.title('2SFS: Few-Shot Accuracy on CIFAR-10 (LayerNorm-tuning + Linear Head)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('2sfs_curve.png', dpi=150, bbox_inches='tight')
plt.show()

KeyError: 100

<Figure size 800x550 with 0 Axes>

---

## Note on fidelity

This reproduces the two-stage spirit of 2SFS (LayerNorm-only Stage 1, frozen-feature linear-head Stage 2) with a simple val-accuracy-peak heuristic for the breakpoint, rather than the paper's more elaborate breakpoint criterion. At K=1 there's no room for a validation split, so Stage 1 runs a fixed small number of epochs instead — worth noting as a limitation if you report this K value.